In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

# Search

> Functionalities to search, and retrieve data from pubmed

In [ ]:
#| default_exp search

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from Bio import Entrez
import os
from datetime import datetime, timedelta, date
from fastcore.all import *
from typing import Union, Optional, Any
from pydantic import BaseModel, ValidationError, model_validator, field_validator


In [ ]:
#| export
from pubmed_lib.data import *
from pubmed_lib.result import *
from pubmed_lib.parser import *

In [ ]:
#| hide
from dotenv import load_dotenv, find_dotenv

In [ ]:
#| hide
load_dotenv(find_dotenv())
email = os.environ.get('EMAIL')
api_key = os.environ.get('API_KEY')

In [ ]:
#| exports

class Search(BaseModel):
    """
    Search class to warp the search and results
    """
    search_tag:str  = 'Title/Abstract' #Tag to specifiy the search, can be any from pubmed, Defaul: Title/Abstract
    retmax:int = 200 #Maximum number of results to be retrieved
    retmode:str ='xml' #Format of the returned data, options are xml, 
    sort:str='relevance' #Way to sort the results
    mindate: int | None = None #Initial data to be search from, year
    maxdate: int | None = None #Final data to be search from, year
    idlist: List[int] | None = None
    email:str | None = None
    api_key:str | None = None
    country: str | None = None
    
    @model_validator(mode='before')
    def validate_email(cls,values:dict )->dict:
        email = get_from_dict_or_env(
            values, "email", "EMAIL"
        )
        values["email"] = email
        
        api_key = get_from_dict_or_env(values, 'api_key', 'API_KEY')
        values['api_key'] = api_key
        return values
        
    @field_validator('search_tag', mode='before')
    @classmethod
    def validate_search_tag(cls, v):
        if not v:
            v = 'Title/Abstract'
        if v not in SEARCH_TAGS.keys():
            raise ValueError(f'Search tag need to be some of {SEARCH_TAGS.keys()}')
        return SEARCH_TAGS[v]
    
     

In [ ]:
#| export

@patch
def _generate_query(
    self:Search,
    query: str, #Query to be search in pubmed
):
    """
    It receive a query and prepare the string to be search, adding all the tags needed
    """

    return f"\"{query}\"{self.search_tag} AND \"{self.country}\"{SEARCH_TAGS['Affiliation']}" if self.country else f"{query}{self.search_tag}"


NameError: name 'patch' is not defined

In [ ]:
#| exports

@patch()
def search(
    self:Search,
    query: str, #Query to be search in pubmed
):
    """
    It receive a query to be searched in pubmed and return the handler of the search
    """
    Entrez.email = self.email
    Entrez.api_key = self.api_key
    query = self._generate_query(query)
    handle = Entrez.esearch(db='pubmed',
                    sort=self.sort,
                    retmax=self.retmax,
                    retmode=self.retmode,
                    term=query,
                    mindate = self.mindate,
                    maxdate =self. maxdate)
    results = Entrez.read(handle)
    return results['IdList']

In [ ]:
search = Search(country='Brasil', mindate=2020, maxdate=2024)

In [ ]:
search

Search(search_tag='Title/Abstract', retmax=200, retmode='xml', sort='relevance', mindate=2020, maxdate=2024, idlist=None, email='elmaturana@gmail.com', api_key='3cfd60b4f78696d27f1c4df78d3fe6f90a09', country='Brasil')

In [ ]:
idlist = search.search('Protein stability')
idlist

['34137435', '33905626', '32492183', '37316640']

In [ ]:
search._generate_query('Protein stability')

'"Protein stability"[tiab] AND "Brasil"[ad]'

In [ ]:
#| export
@patch
def fetch_details(
    self:Search,
    idlist:List[int], #list of pubmedid to be retreived
    ):
    """
    It receive a list of pubmedIds from a search, and retrieve all the details of those publications
    """
    ids = ','.join(idlist)
    handle = Entrez.efetch(db='pubmed',
                           retmode=self.retmode,
                           id=ids)
    results = Entrez.read(handle)
    return results['PubmedArticle']

In [ ]:
#| exports
@patch
def results(
    self:Search,
    query:str, #Term to be queried in pubmed
)->list:
    """
    Method that do the search and retrieve a generator with all the infomration of the articles"""
    results = Results()
    # id_list = self.search(query)
    if id_list := self.search(query):
        articles = self.fetch_details(id_list)
        for article in articles:
            article_dict = parse_paperinfo(article)
            results.append( Result.model_validate(article_dict))
    return results


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()